# AI Job Match Recommendation System
### Hands-on Google Colab: Pretrained Embeddings + Semantic Matching

**Tujuan pembelajaran**
- Memuat dataset lowongan kerja (`CSV`).
- Memahami struktur dan membersihkan data.
- Menggunakan pretrained Sentence Transformer untuk membuat embeddings.
- Merekomendasikan lowongan berdasarkan profil kandidat.
- Memeriksa skill yang cocok dan skill yang belum disebutkan.
- Mengevaluasi rekomendasi secara sederhana.

**Catatan penting**
- Notebook ini tidak melatih model dari nol.
- Skor kemiripan adalah ukuran kemiripan teks, **bukan probabilitas diterima kerja**.
- Skill gap awal berbasis pencocokan kata sederhana; hasilnya perlu diperiksa dan dapat ditingkatkan.
- Jangan gunakan atribut sensitif (misalnya ras, agama, gender, disabilitas, atau usia) untuk memberi peringkat kandidat.


## 1. Install dependencies

Jalankan cell ini terlebih dahulu. Model pretrained akan diunduh saat pertama kali digunakan, sehingga membutuhkan koneksi internet.

In [1]:
!pip -q install sentence-transformers scikit-learn pandas numpy

## 2. Upload dataset CSV

Upload file dataset lowongan kerja kamu, misalnya `JobsFE.csv`. Jika nama file berbeda, kode akan tetap mencoba membaca file CSV yang kamu pilih.

In [2]:
from google.colab import files
import pandas as pd
import numpy as np
import io

uploaded = files.upload()

if not uploaded:
    raise ValueError("Belum ada file yang di-upload.")

csv_name = next(iter(uploaded))
df_jobs = pd.read_csv(io.BytesIO(uploaded[csv_name]))

print(f"File: {csv_name}")
print(f"Ukuran dataset: {df_jobs.shape[0]:,} baris × {df_jobs.shape[1]} kolom")
display(df_jobs.head())


ModuleNotFoundError: No module named 'google.colab'

## 3. Periksa kolom dan kualitas data

Notebook ini mengenali beberapa nama kolom yang umum. Untuk dataset yang sebelumnya dibahas, kolom yang mungkin tersedia antara lain `position`, `job_role_and_duties`, `requisite_skill`, `workplace`, dan `working_mode`. Jika nama kolom berbeda, kamu bisa memilih kolomnya secara manual di cell berikutnya.

In [3]:
print("Daftar kolom:")
for col in df_jobs.columns:
    print("-", col)

display(df_jobs.isna().sum().sort_values(ascending=False).to_frame("missing_values").head(20))


Daftar kolom:
- Job Id
- workplace
- working_mode
- salary
- position
- job_role_and_duties
- requisite_skill
- offer_details


,missing_values
Job Id,0
workplace,0
working_mode,0
salary,0
position,0
job_role_and_duties,0
requisite_skill,0
offer_details,0


## 4. Tentukan kolom yang digunakan

Kode akan mencoba mendeteksi kolom judul pekerjaan, deskripsi/tanggung jawab, skill, lokasi, dan mode kerja. Periksa hasilnya. Jika tidak tepat, ubah variabel pada cell ini.

In [4]:
def find_column(columns, candidates):
    # Pencocokan nama kolom tanpa membedakan huruf besar/kecil
    normalized = {str(c).strip().lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in normalized:
            return normalized[candidate.lower()]
    # Coba pencocokan sebagian nama
    for c in columns:
        cl = str(c).strip().lower()
        if any(candidate.lower() in cl for candidate in candidates):
            return c
    return None

title_col = find_column(df_jobs.columns, [
    "position", "job_title", "title", "job title", "role"
])
description_col = find_column(df_jobs.columns, [
    "job_role_and_duties", "job_description", "description",
    "responsibilities", "duties", "job role"
])
skills_col = find_column(df_jobs.columns, [
    "requisite_skill", "required_skills", "skills", "skill",
    "requirements", "qualifications"
])
location_col = find_column(df_jobs.columns, [
    "workplace", "location", "city", "job_location"
])
mode_col = find_column(df_jobs.columns, [
    "working_mode", "work_mode", "remote_type", "workplace_type"
])

print("Judul pekerjaan:", title_col)
print("Deskripsi:", description_col)
print("Skill:", skills_col)
print("Lokasi:", location_col)
print("Mode kerja:", mode_col)

# Jika deteksi tidak tepat, isi nama kolom secara manual, misalnya:
# title_col = "position"
# description_col = "job_role_and_duties"
# skills_col = "requisite_skill"
# location_col = "workplace"
# mode_col = "working_mode"

if title_col is None:
    raise ValueError("Kolom judul pekerjaan tidak terdeteksi. Atur title_col secara manual.")


Judul pekerjaan: position
Deskripsi: job_role_and_duties
Skill: requisite_skill
Lokasi: workplace
Mode kerja: working_mode


## 5. Cleaning dan membuat teks gabungan

Model akan membandingkan profil kandidat dengan gabungan judul, deskripsi pekerjaan, dan skill yang dibutuhkan.

In [5]:
def safe_text(row, col):
    if col is None:
        return ""
    value = row.get(col, "")
    if pd.isna(value):
        return ""
    return str(value).strip()

df_jobs = df_jobs.copy()

df_jobs["_title"] = df_jobs.apply(lambda r: safe_text(r, title_col), axis=1)
df_jobs["_description"] = df_jobs.apply(lambda r: safe_text(r, description_col), axis=1)
df_jobs["_skills"] = df_jobs.apply(lambda r: safe_text(r, skills_col), axis=1)
df_jobs["_location"] = df_jobs.apply(lambda r: safe_text(r, location_col), axis=1)
df_jobs["_mode"] = df_jobs.apply(lambda r: safe_text(r, mode_col), axis=1)

df_jobs["_job_text"] = (
    "Job title: " + df_jobs["_title"] +
    ". Responsibilities and description: " + df_jobs["_description"] +
    ". Required skills: " + df_jobs["_skills"]
).str.strip()

# Hapus baris tanpa informasi lowongan yang berarti
df_jobs = df_jobs[df_jobs["_job_text"].str.len() > len("Job title: . Responsibilities and description: . Required skills: ")].copy()
df_jobs = df_jobs.drop_duplicates(subset=["_job_text"]).reset_index(drop=True)

print(f"Lowongan setelah cleaning: {len(df_jobs):,}")
display(df_jobs[["_title", "_location", "_mode", "_skills"]].head(10))


Lowongan setelah cleaning: 9,651


,_title,_location,_mode,_skills
0,procurement manager,panama city panama,contract,supplier diversity programs diversity and incl...
1,architectural designer,tunis tunisia,part-time,architectural drafting autocad 2d and 3d model...
2,art teacher,harare zimbabwe,full-time,art education curriculum program development t...
3,environmental consultant,tirana albania,temporary,environmental impact analysis data collection ...
4,art teacher,city of baghdad iraq,temporary,art education curriculum program development t...
5,event planner,andorra la vella andorra,contract,wedding planning venue selection catering and ...
6,architect,asmara eritrea,contract,architectural design software eg autocad revit...
7,family lawyer,port-au-prince haiti,contract,mediation conflict resolution negotiation comm...
8,account manager,seoul north korea,temporary,client relationship management customer servic...
9,business development manager,montevideo uruguay,part-time,market expansion strategies market research sa...


## 6. Load pretrained model

Model default: `paraphrase-MiniLM-L6-v2`, model embedding yang relatif ringan untuk baseline semantic matching berbahasa Inggris.

Alternatif untuk eksperimen:
- `sentence-transformers/all-MiniLM-L6-v2`
- `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2` untuk teks multilingual
- `intfloat/multilingual-e5-small` (memerlukan format query/passsage yang sesuai)

Mulai dengan satu model sebagai baseline, lalu bandingkan hasil secara terkontrol.

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import time

MODEL_NAME = "sentence-transformers/paraphrase-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

start_time = time.time()
job_embeddings = model.encode(
    df_jobs["_job_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)
embedding_seconds = time.time() - start_time

print("Model:", MODEL_NAME)
print("Jumlah embeddings:", len(job_embeddings))
print(f"Waktu membuat embeddings: {embedding_seconds:.1f} detik")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/302 [00:00<?, ?it/s]

Model: sentence-transformers/paraphrase-MiniLM-L6-v2
Jumlah embeddings: 9651
Waktu membuat embeddings: 523.4 detik


## 7. Buat fungsi rekomendasi pekerjaan

Fungsi ini:
1. Mengubah profil kandidat menjadi embedding.
2. Menghitung cosine similarity dengan seluruh lowongan.
3. Mengembalikan lowongan dengan skor tertinggi.
4. Menampilkan lokasi, mode kerja, dan skill yang tercatat pada dataset.

**Interpretasi:** skor lebih tinggi berarti teks lebih mirip menurut model, bukan peluang diterima.

In [7]:
def recommend_jobs(candidate_profile, top_k=5, location_filter=None, mode_filter=None):
    if not isinstance(candidate_profile, str) or not candidate_profile.strip():
        raise ValueError("Masukkan profil kandidat dalam bentuk teks.")

    candidate_embedding = model.encode(
        [candidate_profile],
        normalize_embeddings=True
    )

    scores = cosine_similarity(candidate_embedding, job_embeddings)[0]
    ranked = df_jobs.copy()
    ranked["_match_score"] = scores

    if location_filter:
        ranked = ranked[
            ranked["_location"].str.contains(location_filter, case=False, na=False)
        ]
    if mode_filter:
        ranked = ranked[
            ranked["_mode"].str.contains(mode_filter, case=False, na=False)
        ]

    ranked = ranked.sort_values("_match_score", ascending=False).head(top_k)

    output = pd.DataFrame({
        "Job title": ranked["_title"],
        "Location": ranked["_location"],
        "Work mode": ranked["_mode"],
        "Required skills": ranked["_skills"],
        "Similarity score": ranked["_match_score"].round(3)
    })

    return output.reset_index(drop=True)


## 8. Coba rekomendasi dengan profil kandidat

Ubah profil sesuai minat peserta. Gunakan informasi profesional yang relevan saja.

In [8]:
candidate_profile = """
I have experience with Python, SQL, data analysis, data visualization,
and building machine learning projects. I enjoy turning data into
insights and communicating findings to stakeholders. I am looking for
a junior data analyst, BI analyst, or junior data scientist position.
"""

recommendations = recommend_jobs(candidate_profile, top_k=5)
display(recommendations)


,Job title,Location,Work mode,Required skills,Similarity score
0,business analyst,yaounde cameroon,full-time,data analysis data modeling data visualization...,0.686
1,business analyst,chisinau moldova,contract,data analysis data modeling data visualization...,0.684
2,business analyst,sri jayawardenepura kotte sri lanka,temporary,data analysis data modeling data visualization...,0.684
3,business analyst,kingston jamaica,full-time,data analysis data modeling data visualization...,0.681
4,business analyst,gibraltar gibraltar,full-time,data analysis data modeling data visualization...,0.681


## 9. Skill gap sederhana

Cell ini membandingkan kata kunci skill dari kolom skill lowongan dengan skill yang disebutkan kandidat. Ini hanya baseline berbasis kata/phrase, bukan ekstraksi skill sempurna. Sinonim seperti `ML` dan `machine learning` bisa tidak terdeteksi sebagai hal yang sama.

In [9]:
import re

def normalize_text(text):
    return re.sub(r"[^a-z0-9+#.\s-]", " ", str(text).lower())

def simple_skill_gap(candidate_profile, job_skills):
    candidate_text = normalize_text(candidate_profile)
    # Pecah daftar skill berdasarkan koma, titik koma, atau baris baru
    skill_items = re.split(r"[,;\n|]+", str(job_skills))
    skill_items = [s.strip() for s in skill_items if len(s.strip()) > 1]

    matched, missing = [], []
    for skill in skill_items:
        normalized_skill = normalize_text(skill).strip()
        if not normalized_skill:
            continue
        if normalized_skill in candidate_text:
            matched.append(skill)
        else:
            missing.append(skill)

    return matched, missing

# Analisis skill gap untuk rekomendasi teratas
if len(recommendations):
    first_title = recommendations.iloc[0]["Job title"]
    matching_rows = df_jobs[df_jobs["_title"] == first_title]

    if len(matching_rows):
        job = matching_rows.iloc[0]
        matched, missing = simple_skill_gap(candidate_profile, job["_skills"])

        print("Posisi:", job["_title"])
        print("\nSkill yang terdeteksi cocok:")
        print(matched if matched else "Belum ada kecocokan kata kunci yang terdeteksi.")

        print("\nSkill yang belum disebutkan kandidat:")
        print(missing if missing else "Tidak ada skill yang terdeteksi hilang dari teks profil.")


Posisi: business analyst

Skill yang terdeteksi cocok:
Belum ada kecocokan kata kunci yang terdeteksi.

Skill yang belum disebutkan kandidat:
['business process analysis requirement gathering data analysis communication skills 1 to 11 years ba']


## 10. Eksperimen kelas: bandingkan profil kandidat

Ubah profil kandidat dan amati apakah urutan pekerjaan berubah. Ini membantu peserta memahami personalisasi.

In [10]:
profiles = {
    "Data Analyst": "Python, SQL, Excel, Tableau, data cleaning, dashboards, data visualization, business reporting.",
    "Frontend Developer": "JavaScript, React, HTML, CSS, UI development, responsive web applications, API integration.",
    "Machine Learning Engineer": "Python, machine learning, model evaluation, feature engineering, deployment, APIs, data pipelines."
}

for label, profile in profiles.items():
    print("\n" + "=" * 70)
    print("PROFIL:", label)
    display(recommend_jobs(profile, top_k=3))



PROFIL: Data Analyst


,Job title,Location,Work mode,Required skills,Similarity score
0,data analyst,willemstad curacao,part-time,data analysis and visualization tools eg table...,0.562
1,data analyst,windhoek namibia,intern,data analysis and visualization tools eg table...,0.562
2,data analyst,pretoria south africa,intern,data analysis and visualization tools eg table...,0.562



PROFIL: Frontend Developer


,Job title,Location,Work mode,Required skills,Similarity score
0,ui developer,kigali rwanda,contract,frontend web development html css javascript r...,0.603
1,ui developer,moroni comoros,contract,frontend web development html css javascript r...,0.601
2,ui developer,tripoli libya,intern,frontend web development html css javascript r...,0.600



PROFIL: Machine Learning Engineer


,Job title,Location,Work mode,Required skills,Similarity score
0,data analyst,willemstad curacao,intern,machine learning algorithms and libraries eg s...,0.612
1,data analyst,cairo egypt,temporary,machine learning algorithms and libraries eg s...,0.612
2,data analyst,gibraltar gibraltar,full-time,machine learning algorithms and libraries eg s...,0.612


## 11. Ide pengembangan ke aplikasi full-stack

Setelah notebook berfungsi:
1. Buat form profil kandidat dan filter lokasi/mode kerja.
2. Simpan lowongan dan profil di database (hindari menyimpan CV pribadi tanpa persetujuan).
3. Buat API menggunakan FastAPI atau Next.js.
4. Tampilkan kartu lowongan, alasan rekomendasi, dan skill gap.
5. Evaluasi dengan data berlabel dan tinjau bias.

**Prinsip responsible AI:** rekomendasi harus membantu kandidat menemukan peluang, bukan menjadi keputusan otomatis untuk menolak atau menerima kandidat. Hindari penggunaan atribut sensitif dan uji apakah ranking merugikan kelompok tertentu.